# Waymaker Artemis — Mission 005-E: 응답 계약 v2 재평가

무료 Colab T4에서 저장소 코드(커밋 `f7c751b5172a59155e35ea7750f5ddf75054aaea`)로 공식 `google/gemma-4-E4B-it`(revision `ee0ef60…`)를 NF4로 **한 번** 올린 뒤,
(1) 합성 명세 검증 15건, (2) 개발 후보 20건을 **응답 계약 v2**로 한 번씩 생성합니다.

- 계약 v2와 평가기는 이 실행 전에 고정했습니다. DEV-20 결과를 보고 프롬프트를 바꿔 다시 돌리지 않습니다.
- 생성만 합니다. 채점은 저장소의 `score_dev20.py`로 따로 합니다. 정답·채점 기준은 프롬프트에 들어가지 않습니다.
- held-out 40건은 실행하지 않습니다. 결과는 개발 후보 진단이며 Official Gold Accuracy는 `N/A`입니다.
- 비용: 무료 런타임만 씁니다. 토큰이 필요 없습니다.

In [ ]:
# 1. GPU·VRAM·디스크 확인 (읽기 전용)
import shutil, subprocess, sys
print(sys.version)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

## 2. 무료 실행 환경 확인

1. **런타임 → 런타임 유형 변경**에서 `T4 GPU`인지 봅니다.
2. **리소스 보기**에서 구독 없음·컴퓨팅 단위 미사용 상태인지 봅니다.
3. 둘 다 맞으면 아래 값을 `True`로 바꿉니다.

In [ ]:
FREE_TIER_CONFIRMED = False  # 위 두 가지를 직접 확인한 뒤에만 True로 바꿉니다
assert FREE_TIER_CONFIRMED, '무료 T4 런타임인지 먼저 확인하세요'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
assert gpu == 'Tesla T4', f'예상한 무료 T4가 아닙니다: {gpu!r}'

In [ ]:
# 3. 코드와 패키지 버전 고정
CODE_COMMIT = 'f7c751b5172a59155e35ea7750f5ddf75054aaea'
!git init -q /content/Visable
!git -C /content/Visable remote add origin https://github.com/club-paradiso/Visable.git
# artemis_eval -> procedure_route imports backend/services (standard library only, about 5MB).
!git -C /content/Visable sparse-checkout set experiments/waymaker-ai backend/services
!git -C /content/Visable fetch -q --depth 1 --filter=blob:none origin {CODE_COMMIT}
!git -C /content/Visable checkout -q FETCH_HEAD
head = subprocess.run(['git', '-C', '/content/Visable', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == CODE_COMMIT, f'코드 커밋 불일치: {head}'
%pip install -q "transformers==5.20.2" "bitsandbytes==0.50.2"
import importlib.metadata as md
print({n: md.version(n) for n in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'huggingface-hub', 'tokenizers')})

In [ ]:
# 4. 체크포인트 다운로드(이 셀만 가중치를 받습니다, 약 16GB)와 SHA-256 확인
import json, os
REPO = '/content/Visable/experiments/waymaker-ai'
sys.path[:0] = [REPO, REPO + '/artemis_bootstrap']
policy = json.load(open(REPO + '/configs/artemis_cuda_runtime.json'))
MODEL, REVISION = policy['model']['checkpoint'], policy['model']['revision']
assert (MODEL, REVISION) == ('google/gemma-4-E4B-it', 'ee0ef6023621cff504d758262d4e04895a5af4a2')
assert shutil.disk_usage('/content').free / 1e9 >= policy['limits']['min_free_disk_gb_for_download']
from huggingface_hub import snapshot_download
from pathlib import Path
snapshot = Path(snapshot_download(MODEL, revision=REVISION, token=False))
from waymaker_ai import cuda_policy
verified = cuda_policy.validate_snapshot(snapshot)  # 1~2분
print(verified['weight_integrity'], verified['artifact_integrity']['model.safetensors']['digest'])

In [ ]:
# 5. CUDA 사전 검사. 이후로는 네트워크를 쓰지 않습니다.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import gemma4_cuda_runtime as cuda
check = cuda.preflight()
print('blockers:', check['blockers'], '| version drift:', check['probe']['version_drift'])

In [ ]:
# 6. 모델을 한 번만 올리고 NF4 검증
prepared = cuda.PreparedRun()
print('loaded:', prepared.prepare(), '| blocker:', prepared.blocker)
quant = prepared.header.get('load', {}).get('quantization', {})
print({k: quant.get(k) for k in ('passed', 'quantized_decoder_layers', 'linear4bit_modules', 'quant_types', 'parameter_devices')})

In [ ]:
# 7. 합성 명세 검증 15건 (계약 v2). DEV-20과 섞지 않습니다.
import run_dev20_cuda
SYN_DIR = '/content/artemis_mission005e_synthetic'
print(run_dev20_cuda.main(['--out-dir', SYN_DIR, '--suite', 'synthetic', '--contract', 'v2'], prepared=prepared))

In [ ]:
# 8. 개발 후보 20건 (계약 v2), 같은 모델 로드로 한 번만. 연결이 끊기면 1~6을 다시 실행한 뒤 이 셀만 다시 실행합니다.
DEV_DIR = '/content/artemis_mission005e_dev20_v2'
print(run_dev20_cuda.main(['--out-dir', DEV_DIR, '--cases', 'all', '--contract', 'v2'], prepared=prepared))

In [ ]:
# 9. 결과 해시
import hashlib
for d in (SYN_DIR, DEV_DIR):
    for name in ('generations.jsonl', 'sessions.jsonl'):
        raw = open(f'{d}/{name}', 'rb').read()
        print(d.rsplit('/', 1)[-1], name, len(raw), hashlib.sha256(raw).hexdigest())

In [ ]:
# 10. 결과 내려받기
!cd /content && zip -qr artemis_mission005e.zip artemis_mission005e_synthetic artemis_mission005e_dev20_v2
from google.colab import files
files.download('/content/artemis_mission005e.zip')

## 끝난 뒤

- 두 폴더의 `generations.jsonl`, `sessions.jsonl`을 고치지 않고 `experiments/waymaker-ai/artemis_bootstrap/runs/mission005e/` 아래에 넣고 채점합니다.
- **런타임 → 런타임 연결 해제 및 삭제**로 GPU를 반납합니다.